# 第 10 章 国際貿易と比較優位: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/micro-basics-programming/10-trade-and-comparative-advantage.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
demand_intercept = 100  # 国内市場の需要 Q = a + s_d·P の切片 a
demand_slope = -1       # 需要の傾き s_d（負）
supply_intercept = -10  # 国内市場の供給 Q = c + s_s·P の切片 c
supply_slope = 1        # 供給の傾き s_s（正）
world_price = 40        # 世界価格
tariff = 10             # 関税（1 単位あたり）
q_max = 100             # 図の数量の範囲
p_max = 100             # 図の価格の範囲
# 比較優位: 1 単位を作るのに要る労働（工業・農業）と、国の労働の総量
large_industry, large_agriculture, large_labor = 4, 8, 80
small_industry, small_agriculture, small_labor = 20, 10, 40

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

## 貿易と関税

In [ ]:
from lib.chapter08.market import LinearMarket
from lib.chapter10.trade import (
    free_trade_result,
    gains_from_trade,
    tariff_deadweight_loss,
    tariff_result,
)

market = LinearMarket(demand_intercept, demand_slope, supply_intercept, supply_slope)
r = free_trade_result(market, world_price)
print(f"自由貿易（世界価格 {world_price}）: 需要 {r.quantity_demanded:.2f}、"
      f"供給 {r.quantity_supplied:.2f}、輸入 {r.net_imports:.2f}")
print(f"消費者余剰 {r.consumer_surplus:.2f}、生産者余剰 {r.producer_surplus:.2f}、"
      f"貿易の利益 {gains_from_trade(market, world_price):.2f}")
t = tariff_result(market, world_price, tariff)
print(f"関税 {tariff}: 国内価格 {t.domestic_price:.2f}、輸入 {t.net_imports:.2f}、"
      f"関税収入 {t.tariff_revenue:.2f}")
print(f"関税の死荷重: {tariff_deadweight_loss(market, world_price, tariff):.2f}")

## 比較優位

In [ ]:
from lib.chapter10.comparative import (
    Country,
    comparative_advantage,
    mutually_beneficial_terms,
    opportunity_cost_of_industry,
    ppf_endpoints,
)

large = Country("Large", large_industry, large_agriculture)
small = Country("Small", small_industry, small_agriculture)
print(f"工業の機会費用: 大国 {opportunity_cost_of_industry(large):.2f}、"
      f"小国 {opportunity_cost_of_industry(small):.2f}")
print(f"比較優位: 工業は {comparative_advantage(large, small, 'industry')}、"
      f"農業は {comparative_advantage(large, small, 'agriculture')}")
print(f"フロンティアの両端: 大国 {ppf_endpoints(large, large_labor)}、"
      f"小国 {ppf_endpoints(small, small_labor)}")
low, high = mutually_beneficial_terms(large, small)
print(f"両国が得をする交換比率: {low:.2f} から {high:.2f}")

## 図

In [ ]:
from lib.chapter10.plot import plot_tariff

plot_tariff(market, world_price, tariff, q_max, p_max)

In [ ]:
from lib.chapter10.plot import plot_ppf

plot_ppf(large, large_labor, small, small_labor)

## 試してみよう

- `tariff = 0` にすると、国内価格は 40（世界価格）、輸入は 30 で、関税収入も死荷重も
  0 です。`tariff = 5` なら、国内価格 45、輸入 20、関税収入 100、死荷重 25.0 です
- `tariff = 15` にすると、国内価格は 55（関税なしの均衡価格）で、輸入は 0 です。
  関税収入は 0、死荷重は 225.0（貿易の利益 225 を、すべて失う）。**禁止的な関税**です。
  これ以上上げても、変わりません
- `world_price = 30` にすると、輸入は 50、貿易の利益は 625.0 です。世界価格が、
  国内の均衡価格（55）から離れるほど、貿易の利益は大きくなります
- `small_industry = 6`、`small_agriculture = 10` にすると、小国の工業の機会費用は
  0.6 で、両国が得をする交換比率は、0.5 から 0.6 の**狭い範囲**になります
- `small_industry = 8`、`small_agriculture = 16`（大国と同じ比率）にすると、
  比較優位は、どちらにもなく（None）、貿易の利益の余地が、なくなります。大国は、
  どちらの財でも、絶対優位です